# Mandatory Assignment 01 - Natural Language Processing and Text Analytics

#### Student Numbers: 161612, 160364, 185903, 186634

## Part 2: Linguistic Analysis of a Text Corpus Using spaCy

In [2]:
import pandas as pd
import re
import spacy
from collections import Counter

# -------------------------
# 1) Load Excel and extract SOS column
# -------------------------

df = pd.read_excel("sample.xlsx", header=8)

# Clean column names (make sure to remove extra spaces)
df.columns = df.columns.astype(str).str.strip()

print("Columns:", df.columns.tolist())

col = "SOS Tweet / SOS Message"

# Get all non-empty SOS messages as well
texts = df[col].dropna().astype(str).tolist()

print("Number of messages:", len(texts))
print("Example:", texts[0])

Columns: ['Unique_Id', 'Se. No.', 'SOS Tweet / SOS Message', 'Least Urgent\n(Score 1)', '(Score 2)', '(Score 3)', '(Score 4)', 'Most Urgent\n(Score 5)', 'Comment']
Number of messages: 100
Example: my relative B************n who is from bagmara jharkhand is tested positive for corona and is admitted in recovery nourshing home bardmaan West Bengal.He is not getting sufficient treatment and his health is getting worse. We need an urgent ventilation bed. HELP [Redacted Mention]


## 1) Prepare the corpus

In [3]:
# -------------------------
# 2) Two different preprocessing versions
# -------------------------

# A) POS preprocessing: cleaner text for POS stats

def preprocess_pos(text: str) -> str:
    
    # make everything lower cases
    text = str(text).lower()

    # remove hashtags
    text = re.sub(r"#\w+", " ", text)

    # remove any tokens
    text = re.sub(r"\b[\d\*]{6,}\b", " ", text)

    # fix missing space after punctuation and normalize dots
    text = re.sub(r"([.!?])([a-z])", r"\1 \2", text)
    text = re.sub(r"\.{2,}", ".", text)

    # remove corrupted emoji
    text = text.encode("ascii", "ignore").decode()

    # remove leftover punctuation sequences
    text = re.sub(r"\s*[-,:]\s*", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text
    
# B) NER preprocessing: keep more original structure for better entity detection

def preprocess_ner(text: str) -> str:

    # Remove bracketed placeholders
    text = re.sub(r"\[.*?\]", " ", text)

    # Remove URLs 
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Normalize whitespace and trim ends
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Create both cleaned versions
clean_texts_pos = [preprocess_pos(t) for t in texts]
clean_texts_ner = [preprocess_ner(t) for t in texts]

# Print first 5 POS-cleaned messages to check
for i in range(3):
    print(f"\n--- Message {i+1} (ORIGINAL) ---")
    print(texts[i])

    print("--- POS-cleaned ---")
    print(clean_texts_pos[i])

    print("--- NER-cleaned ---")
    print(clean_texts_ner[i])


--- Message 1 (ORIGINAL) ---
my relative B************n who is from bagmara jharkhand is tested positive for corona and is admitted in recovery nourshing home bardmaan West Bengal.He is not getting sufficient treatment and his health is getting worse. We need an urgent ventilation bed. HELP [Redacted Mention]
--- POS-cleaned ---
my relative b n who is from bagmara jharkhand is tested positive for corona and is admitted in recovery nourshing home bardmaan west bengal. he is not getting sufficient treatment and his health is getting worse. we need an urgent ventilation bed. help [redacted mention]
--- NER-cleaned ---
my relative B************n who is from bagmara jharkhand is tested positive for corona and is admitted in recovery nourshing home bardmaan West Bengal.He is not getting sufficient treatment and his health is getting worse. We need an urgent ventilation bed. HELP

--- Message 2 (ORIGINAL) ---
[Redacted Mention] [Redacted Mention] we need urgently oxygen sylinder in Nadiad fo

## 2) POS tagging + NER in spaCy

In [5]:
nlp = spacy.load("en_core_web_sm")

pos_counts = Counter()
ner_counts = Counter()

verb_counts = Counter()   # count actual verbs (words)
noun_counts = Counter()   # count actual nouns (words)

for doc in nlp.pipe(clean_texts_pos, batch_size=50):

    for token in doc:
        # Skip stopwords, punctuation, spaces to focus on content words
        if token.is_stop or token.is_punct or token.is_space:
            continue

        # Count POS tag types
        pos_counts[token.pos_] += 1

        # Collect nouns and verbs
        if token.pos_ == "VERB":
            verb_counts[token.lemma_] += 1   # lemma = base form (need/needed -> need)
        if token.pos_ == "NOUN":
            noun_counts[token.lemma_] += 1

# ---- NER uses NER-cleaned text ----
for doc in nlp.pipe(clean_texts_ner, batch_size=50):
    for ent in doc.ents:
        ner_counts[ent.label_] += 1


# -----------------------------------
# 3) Results: POS + NER (counts + %)
# -----------------------------------

print("\n====== POS Results (POS-cleaned) =======")
total_pos = sum(pos_counts.values())
for tag, c in pos_counts.most_common(10):
    print(f"{tag:>6} : {c:>4}  ({(c/total_pos)*100:.1f}%)")

print("\n====== NER Results (NER-cleaned) =======")
total_ner = sum(ner_counts.values())
for label, c in ner_counts.most_common(10):
    print(f"{label:>10} : {c:>4}  ({(c/total_ner)*100:.1f}%)")


====== POS Results (POS-cleaned) =======
  NOUN : 1105  (45.1%)
  VERB :  533  (21.7%)
 PROPN :  344  (14.0%)
   NUM :  184  (7.5%)
   ADJ :  174  (7.1%)
   ADV :   34  (1.4%)
  INTJ :   20  (0.8%)
 CCONJ :   14  (0.6%)
     X :   10  (0.4%)
   AUX :    9  (0.4%)

====== NER Results (NER-cleaned) =======
  CARDINAL :  114  (26.0%)
    PERSON :  110  (25.1%)
       ORG :  104  (23.7%)
       GPE :   42  (9.6%)
      DATE :   37  (8.4%)
     MONEY :   10  (2.3%)
   PRODUCT :    5  (1.1%)
   PERCENT :    3  (0.7%)
      NORP :    3  (0.7%)
       LAW :    2  (0.5%)


In [7]:
# -------------------------
# 4) Analysis
# -------------------------

# Noun-to-Verb ratio
total_nouns = pos_counts["NOUN"]
total_verbs = pos_counts["VERB"]
ratio = total_nouns / total_verbs if total_verbs != 0 else None

print("\n------ Analysis ------")
print("Total NOUN:", total_nouns)
print("Total VERB:", total_verbs)
print("NOUN:VERB ratio:", round(ratio, 2) if ratio else "undefined")

print("\nTop 10 most frequent VERBS (lemmas):")
print(verb_counts.most_common(10))

print("\nTop 10 most frequent NOUNS (lemmas):")

 
# ---- 1) POS STRUCTURE METRICS ----

# Total number of counted POS tokens
total_pos = sum(pos_counts.values())

# Get some key POS tag counts
noun_count  = pos_counts.get("NOUN", 0)
verb_count  = pos_counts.get("VERB", 0)
adj_count   = pos_counts.get("ADJ", 0)
propn_count = pos_counts.get("PROPN", 0)

# Noun-to-Verb ratio
noun_verb_ratio = (noun_count / verb_count) if verb_count != 0 else None

print("\n================ POS STRUCTURE METRICS ================")
print("Total POS tokens counted:", total_pos)
print("Total NOUN:", noun_count)
print("Total VERB:", verb_count)
print("Total ADJ :", adj_count)
print("Total PROPN:", propn_count)

# Print ratio + rates
print("NOUN:VERB ratio:", round(noun_verb_ratio, 2) if noun_verb_ratio else "undefined")
print("ADJ rate (%):", round((adj_count / total_pos) * 100, 2) if total_pos else 0)
print("PROPN rate (%):", round((propn_count / total_pos) * 100, 2) if total_pos else 0)


# ----- 2) TOP VERBS -----

print("\n================ TOP VERBS (LEMMA) ================")
# verb_counts should contain base forms: need/needed -> need
for v, c in verb_counts.most_common(15):
    print(f"{v:>12} : {c}")


# ---- 3) URGENCY / REQUEST KEYWORDS ----

# Simple keyword list that often appears in SOS messages
urgent_words = ["urgent", "immediately", "asap", "critical", "help", "required", "need"]

urgent_counts = Counter()

# Count how many messages contain each urgent word
for msg in clean_texts_ner:
    for w in urgent_words:
        if w in msg:
            urgent_counts[w] += 1

print("\n================ URGENCY KEYWORDS ================")
for w, c in urgent_counts.most_common():
    print(f"{w:>12} : {c}")
    

# ---- 4) NER DISTRIBUTION (COUNTS + %) ----

total_ner = sum(ner_counts.values())

print("\n================ NER LABEL DISTRIBUTION ================")
print("Total entities found:", total_ner)

# Show top 10 NER labels with percentages
for label, c in ner_counts.most_common(10):
    pct = (c / total_ner) * 100 if total_ner else 0
    print(f"{label:>10} : {c:>4}  ({pct:.1f}%)")


# ---- 5) TOP LOCATIONS, ORGANIZATIONS AND PEOPLE ----

gpe_entities = Counter()
org_entities = Counter()
person_entities = Counter()

# Use the NER-cleaned texts for better entity detection
for doc in nlp.pipe(clean_texts_ner, batch_size=50):
    for ent in doc.ents:
        if ent.label_ == "GPE":
            gpe_entities[ent.text.lower()] += 1
        elif ent.label_ == "ORG":
            org_entities[ent.text.lower()] += 1
        elif ent.label_ == "PERSON":
            person_entities[ent.text.lower()] += 1

print("\n================ TOP GPE (LOCATIONS) ================")
print(gpe_entities.most_common(10))

print("\n================ TOP ORG (ORGANIZATIONS/HOSPITALS) ================")
print(org_entities.most_common(10))

print("\n================ TOP PERSON (PEOPLE NAMES) ================")
print(person_entities.most_common(10))


# ---- 6) MESSAGE LENGTH ANALYSIS (word counts) ----

# Calculate number of words per message
lengths = [len(msg.split()) for msg in clean_texts_ner]

min_len = min(lengths) if lengths else 0
max_len = max(lengths) if lengths else 0
avg_len = (sum(lengths) / len(lengths)) if lengths else 0

print("\n================ MESSAGE LENGTH ================")
print("Min words:", min_len)
print("Max words:", max_len)
print("Average words:", round(avg_len, 2))

# Print 5 longest messages
longest_idx = sorted(range(len(lengths)), key=lambda i: lengths[i], reverse=True)[:5]

print("\n5 LONGEST MESSAGES (word_count -> preview):")
for i in longest_idx:
    preview = clean_texts_ner[i][:160]  # show only first 160 characters
    print(lengths[i], "->", preview, "...")


# ---- 7) TEMPLATE PATTERN ----

# Many SOS messages include semi-structured fields like "name:", "age:", "location:"
fields = ["name:", "age:", "location:", "contact:", "patient:", "icu", "oxygen", "bed", "ventilator"]

field_counts = Counter()

# Count how many messages contain each pattern
for msg in clean_texts_ner:
    for f in fields:
        if f in msg:
            field_counts[f] += 1

print("\n================ TEMPLATE PATTERNS ================")
for f, c in field_counts.most_common():
    print(f"{f:>12} : {c}")


------ Analysis ------
Total NOUN: 1105
Total VERB: 533
NOUN:VERB ratio: 2.07

Top 10 most frequent VERBS (lemmas):
[('redact', 298), ('need', 45), ('help', 41), ('require', 21), ('admit', 11), ('spo2', 6), ('get', 4), ('score', 4), ('covid', 4), ('ve', 4)]

Top 10 most frequent NOUNS (lemmas):

================ POS STRUCTURE METRICS ================
Total POS tokens counted: 2451
Total NOUN: 1105
Total VERB: 533
Total ADJ : 174
Total PROPN: 344
NOUN:VERB ratio: 2.07
ADJ rate (%): 7.1
PROPN rate (%): 14.04

================ TOP VERBS (LEMMA) ================
      redact : 298
        need : 45
        help : 41
     require : 21
       admit : 11
        spo2 : 6
         get : 4
       score : 4
       covid : 4
          ve : 4
        lead : 3
     amplify : 3
     contact : 3
        want : 3
        drop : 2

================ URGENCY KEYWORDS ================
        help : 50
        need : 36
      urgent : 25
    critical : 10
    required : 9
 immediately : 3

==============